In [0]:
from pyspark.sql.functions import *;

In [0]:
spark.conf.set(
    "fs.azure.account.key.vehicletheftstorageacc.blob.core.windows.net",
    dbutils.secrets.get('dbScope', 'storageaccSecret')
)

In [0]:
display(dbutils.fs.ls("wasbs://bronze@vehicletheftstorageacc.blob.core.windows.net/"))

path,name,size,modificationTime
wasbs://bronze@vehicletheftstorageacc.blob.core.windows.net/locations.csv,locations.csv,732,1776784609000
wasbs://bronze@vehicletheftstorageacc.blob.core.windows.net/make_details.csv,make_details.csv,2993,1776784624000
wasbs://bronze@vehicletheftstorageacc.blob.core.windows.net/stolen_vehicles.csv,stolen_vehicles.csv,226866,1776784637000
wasbs://bronze@vehicletheftstorageacc.blob.core.windows.net/stolen_vehicles_db_data_dictionary.csv,stolen_vehicles_db_data_dictionary.csv,866,1776784657000


In [0]:
display(dbutils.fs.ls("wasbs://silver@vehicletheftstorageacc.blob.core.windows.net/"))

[]

In [0]:
display(dbutils.fs.ls("wasbs://gold@vehicletheftstorageacc.blob.core.windows.net/"))

[]

In [0]:
location_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("wasbs://bronze@vehicletheftstorageacc.blob.core.windows.net/locations.csv")

make_details_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("wasbs://bronze@vehicletheftstorageacc.blob.core.windows.net/make_details.csv")

stolen_vehicles_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("wasbs://bronze@vehicletheftstorageacc.blob.core.windows.net/stolen_vehicles.csv")

database_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("wasbs://bronze@vehicletheftstorageacc.blob.core.windows.net/stolen_vehicles_db_data_dictionary.csv")

In [0]:
location_df.show()

+-----------+------------------+-----------+----------+-------+
|location_id|            region|    country|population|density|
+-----------+------------------+-----------+----------+-------+
|        101|         Northland|New Zealand|   201,500|  16.11|
|        102|          Auckland|New Zealand| 1,695,200| 343.09|
|        103|           Waikato|New Zealand|   513,800|   21.5|
|        104|     Bay of Plenty|New Zealand|   347,700|   28.8|
|        105|          Gisborne|New Zealand|    52,100|   6.21|
|        106|       Hawke's Bay|New Zealand|   182,700|  12.92|
|        107|          Taranaki|New Zealand|   127,300|  17.55|
|        108|Manawatū-Whanganui|New Zealand|   258,200|  11.62|
|        109|        Wellington|New Zealand|   543,500|  67.52|
|        110|            Tasman|New Zealand|    58,700|    6.1|
|        111|            Nelson|New Zealand|    54,500| 129.15|
|        112|       Marlborough|New Zealand|    51,900|   4.94|
|        113|        West Coast|New Zeal

In [0]:
make_details_df.show()

+-------+-------------+---------+
|make_id|    make_name|make_type|
+-------+-------------+---------+
|    501|Aakron Xpress| Standard|
|    502|         ADLY| Standard|
|    503|        Alpha| Standard|
|    504|        Anglo| Standard|
|    505|      Aprilia| Standard|
|    506|        Atlas| Standard|
|    507|         Audi| Standard|
|    508|       Bailey| Standard|
|    509|      Bedford| Standard|
|    510|      Benelli| Standard|
|    511|      Bentley|   Luxury|
|    512|          BMW|   Luxury|
|    513|       Bricon| Standard|
|    514|      Briford| Standard|
|    515|        Buell| Standard|
|    516|      Buffalo| Standard|
|    517|     Cadillac|   Luxury|
|    518|       Can-Am| Standard|
|    519|      Caravan| Standard|
|    520|  Caterpillar| Standard|
+-------+-------------+---------+
only showing top 20 rows


In [0]:
stolen_vehicles_df.show()

+----------+------------+-------+----------+--------------------+------+-----------+-----------+
|vehicle_id|vehicle_type|make_id|model_year|        vehicle_desc| color|date_stolen|location_id|
+----------+------------+-------+----------+--------------------+------+-----------+-----------+
|         1|     Trailer|    623|      2021|            BST2021D|Silver| 2021-11-05|        102|
|         2|Boat Trailer|    623|      2021| OUTBACK BOATS FT470|Silver| 2021-12-13|        105|
|         3|Boat Trailer|    623|      2021|          ASD JETSKI|Silver| 2022-02-13|        102|
|         4|     Trailer|    623|      2021|             MSC 7X4|Silver| 2021-11-13|        106|
|         5|     Trailer|    623|      2018|           D-MAX 8X5|Silver| 2022-01-10|        102|
|         6|    Roadbike|    636|      2005|             YZF-R6T| Black| 2021-12-31|        102|
|         7|     Trailer|    623|      2021|    CAAR TRANSPORTER|Silver| 2021-11-12|        114|
|         8|Boat Trailer|    6

In [0]:
database_df.show()

+---------------+------------+--------------------+
|          Table|       Field|         Description|
+---------------+------------+--------------------+
|stolen_vehicles|  vehicle_id|Unique ID of a st...|
|stolen_vehicles|vehicle_type|     Type of vehicle|
|stolen_vehicles|     make_id|Matches make_id i...|
|stolen_vehicles|  model_year|Model year of veh...|
|stolen_vehicles|vehicle_desc|Description of ve...|
|stolen_vehicles|       color|    Color of vehicle|
|stolen_vehicles| date_stolen|Date the vehicle ...|
|stolen_vehicles| location_id|Matches location_...|
|   make_details|     make_id|Unique ID of the ...|
|   make_details|   make_name|    Name of the make|
|   make_details|   make_type|Type of make (Sta...|
|      locations| location_id|Unique ID of the ...|
|      locations|      region|  Name of the region|
|      locations|     country|Country where the...|
|      locations|  population|Population of the...|
|      locations|     density|Density of the re...|
+-----------

In [0]:
location_df.printSchema()

root
 |-- location_id: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- country: string (nullable = true)
 |-- population: string (nullable = true)
 |-- density: double (nullable = true)



In [0]:
location_df=location_df.withColumn("Population",regexp_replace(col("Population"),",","").cast("integer"))

In [0]:
location_df.show()

+-----------+------------------+-----------+----------+-------+
|location_id|            region|    country|Population|density|
+-----------+------------------+-----------+----------+-------+
|        101|         Northland|New Zealand|    201500|  16.11|
|        102|          Auckland|New Zealand|   1695200| 343.09|
|        103|           Waikato|New Zealand|    513800|   21.5|
|        104|     Bay of Plenty|New Zealand|    347700|   28.8|
|        105|          Gisborne|New Zealand|     52100|   6.21|
|        106|       Hawke's Bay|New Zealand|    182700|  12.92|
|        107|          Taranaki|New Zealand|    127300|  17.55|
|        108|Manawatū-Whanganui|New Zealand|    258200|  11.62|
|        109|        Wellington|New Zealand|    543500|  67.52|
|        110|            Tasman|New Zealand|     58700|    6.1|
|        111|            Nelson|New Zealand|     54500| 129.15|
|        112|       Marlborough|New Zealand|     51900|   4.94|
|        113|        West Coast|New Zeal

In [0]:
make_details_df.printSchema()

root
 |-- make_id: integer (nullable = true)
 |-- make_name: string (nullable = true)
 |-- make_type: string (nullable = true)



In [0]:
location_df=location_df.toDF(*[Column.lower().replace(" ","_") for Column in location_df.columns])

make_details_df=make_details_df.toDF(*[Column.lower().replace(" ","_") for Column in make_details_df.columns])

stolen_vehicles_df=stolen_vehicles_df.toDF(*[Column.lower().replace(" ","_") for Column in stolen_vehicles_df.columns])

database_df=database_df.toDF(*[Column.lower().replace(" ","_") for Column in database_df.columns])

# column=location_df.columns
# new_column=[]

# for col in column:
#     clean_column=col.replace(" ", "_").lowwer()
#     new_column.append(clean_column)
# location_df=location_df.toDF(*new_column)
location_df.show(2)
make_details_df.show(2)
stolen_vehicles_df.show(2)
database_df.show(2)

+-----------+---------+-----------+----------+-------+
|location_id|   region|    country|population|density|
+-----------+---------+-----------+----------+-------+
|        101|Northland|New Zealand|    201500|  16.11|
|        102| Auckland|New Zealand|   1695200| 343.09|
+-----------+---------+-----------+----------+-------+
only showing top 2 rows
+-------+-------------+---------+
|make_id|    make_name|make_type|
+-------+-------------+---------+
|    501|Aakron Xpress| Standard|
|    502|         ADLY| Standard|
+-------+-------------+---------+
only showing top 2 rows
+----------+------------+-------+----------+-------------------+------+-----------+-----------+
|vehicle_id|vehicle_type|make_id|model_year|       vehicle_desc| color|date_stolen|location_id|
+----------+------------+-------+----------+-------------------+------+-----------+-----------+
|         1|     Trailer|    623|      2021|           BST2021D|Silver| 2021-11-05|        102|
|         2|Boat Trailer|    623| 

In [0]:
stolen_vehicles_df=stolen_vehicles_df.withColumnRenamed("vvhicle_id","vehicle_id")
stolen_vehicles_df.show(2)

+----------+------------+-------+----------+-------------------+------+-----------+-----------+
|vehicle_id|vehicle_type|make_id|model_year|       vehicle_desc| color|date_stolen|location_id|
+----------+------------+-------+----------+-------------------+------+-----------+-----------+
|         1|     Trailer|    623|      2021|           BST2021D|Silver| 2021-11-05|        102|
|         2|Boat Trailer|    623|      2021|OUTBACK BOATS FT470|Silver| 2021-12-13|        105|
+----------+------------+-------+----------+-------------------+------+-----------+-----------+
only showing top 2 rows


In [0]:
location_df.write.option("header","true").mode("overwrite").csv("wasbs://silver@vehicletheftstorageacc.blob.core.windows.net/location.csv")

make_details_df.write.option("header","true").csv("wasbs://silver@vehicletheftstorageacc.blob.core.windows.net/make_details.csv")

stolen_vehicles_df.write.option("header","true").csv("wasbs://silver@vehicletheftstorageacc.blob.core.windows.net/stolen_vehicles.csv")

database_df.write.option("header","true").csv("wasbs://silver@vehicletheftstorageacc.blob.core.windows.net/database.csv")

In [0]:
null_count_location=location_df.select([sum(when(col(column).isNull(),1).otherwise(0)).alias(column) for column in location_df.columns])

null_count_make_details=make_details_df.select([sum(when(col(column).isNull(),1).otherwise(0)).alias(column) for column in make_details_df.columns])

null_count_stolen_vehicles=stolen_vehicles_df.select([sum(when(col(column).isNull(),1).otherwise(0)).alias(column) for column in stolen_vehicles_df.columns])

null_count_database=database_df.select([sum(when(col(column).isNull(),1).otherwise(0)).alias(column) for column in database_df.columns])

null_count_location.show()
null_count_make_details.show()
null_count_stolen_vehicles.show()
null_count_database.show()

+-----------+------+-------+----------+-------+
|location_id|region|country|population|density|
+-----------+------+-------+----------+-------+
|          0|     0|      0|         0|      0|
+-----------+------+-------+----------+-------+

+-------+---------+---------+
|make_id|make_name|make_type|
+-------+---------+---------+
|      0|        0|        0|
+-------+---------+---------+

+----------+------------+-------+----------+------------+-----+-----------+-----------+
|vehicle_id|vehicle_type|make_id|model_year|vehicle_desc|color|date_stolen|location_id|
+----------+------------+-------+----------+------------+-----+-----------+-----------+
|         0|          26|     15|        15|          33|   15|          0|          0|
+----------+------------+-------+----------+------------+-----+-----------+-----------+

+-----+-----+-----------+
|table|field|description|
+-----+-----+-----------+
|    0|    0|          0|
+-----+-----+-----------+



In [0]:
stolen_vehicles_df.printSchema()

root
 |-- vehicle_id: integer (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- make_id: integer (nullable = true)
 |-- model_year: integer (nullable = true)
 |-- vehicle_desc: string (nullable = true)
 |-- color: string (nullable = true)
 |-- date_stolen: date (nullable = true)
 |-- location_id: integer (nullable = true)



In [0]:
stolen_vehicles_df=stolen_vehicles_df.fillna({
    "vehicle_type":"No data",
    "make_id":0,
    "model_year":0,
    "vehicle_desc":"No data",
    "color":"No data",
})

In [0]:
stolen_vehicles_df.tail(5)

[Row(vehicle_id=4549, vehicle_type='No data', make_id=0, model_year=0, vehicle_desc='No data', color='No data', date_stolen=datetime.date(2022, 2, 18), location_id=102),
 Row(vehicle_id=4550, vehicle_type='No data', make_id=0, model_year=0, vehicle_desc='No data', color='No data', date_stolen=datetime.date(2022, 2, 14), location_id=109),
 Row(vehicle_id=4551, vehicle_type='No data', make_id=0, model_year=0, vehicle_desc='No data', color='No data', date_stolen=datetime.date(2022, 3, 9), location_id=102),
 Row(vehicle_id=4552, vehicle_type='No data', make_id=0, model_year=0, vehicle_desc='No data', color='No data', date_stolen=datetime.date(2022, 3, 7), location_id=109),
 Row(vehicle_id=4553, vehicle_type='No data', make_id=0, model_year=0, vehicle_desc='No data', color='No data', date_stolen=datetime.date(2022, 3, 14), location_id=102)]

In [0]:
null_count_stolen_vehicles=stolen_vehicles_df.select([sum(when(col(column).isNull(),1).otherwise(0)).alias(column) for column in stolen_vehicles_df.columns])
null_count_stolen_vehicles.show()



+----------+------------+-------+----------+------------+-----+-----------+-----------+
|vehicle_id|vehicle_type|make_id|model_year|vehicle_desc|color|date_stolen|location_id|
+----------+------------+-------+----------+------------+-----+-----------+-----------+
|         0|           0|      0|         0|           0|    0|          0|          0|
+----------+------------+-------+----------+------------+-----+-----------+-----------+



In [0]:
stolen_vehicles_df.show(5)

+----------+------------+-------+----------+-------------------+------+-----------+-----------+
|vehicle_id|vehicle_type|make_id|model_year|       vehicle_desc| color|date_stolen|location_id|
+----------+------------+-------+----------+-------------------+------+-----------+-----------+
|         1|     Trailer|    623|      2021|           BST2021D|Silver| 2021-11-05|        102|
|         2|Boat Trailer|    623|      2021|OUTBACK BOATS FT470|Silver| 2021-12-13|        105|
|         3|Boat Trailer|    623|      2021|         ASD JETSKI|Silver| 2022-02-13|        102|
|         4|     Trailer|    623|      2021|            MSC 7X4|Silver| 2021-11-13|        106|
|         5|     Trailer|    623|      2018|          D-MAX 8X5|Silver| 2022-01-10|        102|
+----------+------------+-------+----------+-------------------+------+-----------+-----------+
only showing top 5 rows


In [0]:
location_df.write.option("header","true").mode("overwrite").csv("wasbs://gold@vehicletheftstorageacc.blob.core.windows.net/location.csv")

make_details_df.write.option("header","true").csv("wasbs://gold@vehicletheftstorageacc.blob.core.windows.net/make_details.csv")

stolen_vehicles_df.write.option("header","true").csv("wasbs://gold@vehicletheftstorageacc.blob.core.windows.net/stolen_vehicles.csv")

database_df.write.option("header","true").csv("wasbs://gold@vehicletheftstorageacc.blob.core.windows.net/database.csv")

In [0]:
location_df.createOrReplaceTempView("location")
make_details_df.createOrReplaceTempView("make_details")
stolen_vehicles_df.createOrReplaceTempView("stolen_vehicles")
database_df.createOrReplaceTempView("database") 

In [0]:
%sql
select model_year, count(*) as number_of_vehicles_stolen from stolen_vehicles
group by model_year
order by number_of_vehicles_stolen  desc


model_year,number_of_vehicles_stolen
2005,347
2006,333
2007,251
2004,238
2008,190
2002,181
2003,173
1998,159
1996,156
2001,152
